# Mar 10/11 · 11:30 — TP v0.7 : historique des rapports et archivage

Mar 10/11, TP de 12:00 à 12:50. Correction à 12:50, TP fini ou non. La référence v0.7 est publiée à 13:00.

## Objectif

Pendant l'astreinte, la question est souvent « depuis quand ? ». `fleetcheck` garde un historique :

- chaque exécution écrit son rapport dans `reports/report-<date>.json` au lieu d'écraser `report.json` ;
- les rapports de plus de `archive_after_days` jours partent dans une archive zip, puis disparaissent du dossier.

## Mode d'emploi du notebook

- Copiez ce notebook à la racine de votre dépôt, à côté de `fleetcheck.py`. Le code s'écrit dans `fleetcheck.py`, ouvert dans un onglet de JupyterLab ; le notebook sert à essayer et à vérifier chaque étape.
- Exécutez d'abord la cellule d'outils ci-dessous. Chaque vérification affiche `OK`, ou `KO` avec ce qui était attendu.
- `%run -n fleetcheck.py` charge les fonctions du fichier enregistré sans exécuter le bloc `if __name__ == "__main__":`. N'appelez pas `main()` dans le notebook.
- `run(...)` lance le script comme dans un terminal et renvoie `r.stdout`, `r.stderr` et `r.returncode`.
- Les vérifications de `write_report` et d'`archive_old` travaillent dans un dossier temporaire, supprimé à la fin de la cellule : elles ne touchent pas à votre `reports/`. Seule la vérification de l'étape 3 utilise votre `reports/`, comme le ferait un vrai run.
- **Prédire** et **Justifier** marquent les moments clés du TP. Répondez dans la cellule `# Ma réponse :` qui suit, **avant** d'exécuter la suite.
- Rendu en fin de TP : menu *Kernel > Restart Kernel and Run All Cells*, puis commit et push de `fleetcheck.py`, `config.json`, `requirements.txt`, `.gitignore`, `servers.csv`, `servers-broken.csv` et de ce notebook, sans `reports/`.

In [ ]:
# Outils du TP : exécuter cette cellule en premier, sans la modifier.
import os
import subprocess
import sys


def check(obtenu, attendu):
    if obtenu == attendu:
        print("OK")
    elif isinstance(attendu, str) and "\n" in attendu:
        print(f"KO, obtenu :\n{obtenu}\nattendu :\n{attendu}")
    else:
        print(f"KO\n  obtenu  : {obtenu!r}\n  attendu : {attendu!r}")


def check_error(fonction, *args, attendu):
    try:
        fonction(*args)
    except Exception as e:
        check(f"{type(e).__name__}: {e}", attendu)
    else:
        print(f"KO, aucune exception levée. Attendu : {attendu}")


def run(*args):
    """Lance fleetcheck.py comme dans un terminal (vu en v0.8 : subprocess)."""
    r = subprocess.run([sys.executable, "fleetcheck.py", *args], capture_output=True,
                       encoding="utf-8", env={**os.environ, "PYTHONUTF8": "1", "NO_COLOR": "1",
                                              "PYTHONWARNDEFAULTENCODING": "1"})
    print(r.stdout, end="")
    print(r.stderr, end="", file=sys.stderr)
    print(f"[code de sortie : {r.returncode}]")
    if "EncodingWarning" in r.stderr:
        print("KO : un open(), read_text() ou subprocess text=True sans encoding= (ligne indiquée dans l'EncodingWarning ci-dessus)")
    return r

## Point de départ

Votre `fleetcheck.py` de v0.6, avec `config.json`, `requirements.txt` et les deux inventaires.

En retard ou en panne : copiez dans votre dépôt les fichiers de `reference/` au tag `v0.6` du dépôt public.

`config.json` contient déjà, depuis v0.3, les deux réglages de cette séance :

```text
"reports_dir": "reports",
"archive_after_days": 7
```

Vérifiez avant de commencer :

In [ ]:
RESUME = """9 serveurs valides, 0 invalides
par env : prod 7, staging 2
par rôle : web 3, api 3, worker 1, cache 1, db 1
prod/web : web-1, web-2
"""

import json
from pathlib import Path

config = json.loads(Path("config.json").read_text(encoding="utf-8"))
check((config["reports_dir"], config["archive_after_days"]), ("reports", 7))
r = run()
check((r.stdout, r.returncode), (RESUME, 0))

## Étapes

| Fin | Étape |
|---|---|
| 12:15 | 1. `write_report(report, reports_dir)` : un rapport horodaté par exécution |
| 12:40 | 2. `archive_old(reports_dir, days)` : archiver puis supprimer les rapports anciens |
| 12:50 | 3. `main()` archive après chaque écriture |

À 12:25, si plus d'un tiers de la salle n'a pas fini l'étape 1, l'enseignant écrit `write_report` au tableau ; l'étape 3 passe en devoir.

### 1. `write_report` — fin 12:15

`write_report(report, reports_dir)` :

- crée `reports_dir` et ses parents s'ils manquent (`Path.mkdir(parents=True, exist_ok=True)`) ;
- écrit le rapport dans `report-<date>.json`, date au format `%Y-%m-%dT%H%M%S` : `f"report-{datetime.now():%Y-%m-%dT%H%M%S}.json"` ;
- écrit le JSON par `Path.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")` ;
- renvoie le `Path` du fichier écrit.

Dans `main()`, `write_report(report, config["reports_dir"])` remplace l'écriture de `report.json` ; le log `INFO` donne le chemin renvoyé. Ajoutez `reports/` au `.gitignore` de votre dépôt : un rapport est un résultat, pas une source.

Deux exécutions dans la même seconde écrivent le même nom : la seconde écrase la première. Les vérifications attendent une seconde entre deux runs.

**Justifier** — Pourquoi `%Y-%m-%dT%H%M%S` plutôt que `%d-%m-%Y %H:%M:%S` ?

In [ ]:
# Ma réponse :

Vérification, dans un dossier temporaire : le dossier `sub/reports` est créé, le nom suit le format, le contenu est le rapport.

In [ ]:
import re
import tempfile

%run -n fleetcheck.py
with tempfile.TemporaryDirectory() as tmp:
    chemin = write_report({"summary": {}}, Path(tmp) / "sub" / "reports")
    check((chemin.exists(), chemin.parent.name), (True, "reports"))
    check(re.fullmatch(r"report-\d{4}-\d{2}-\d{2}T\d{6}\.json", chemin.name) is not None, True)
    check(json.loads(chemin.read_text(encoding="utf-8")), {"summary": {}})

Vérification : un run ajoute un rapport dans `reports/`, et `reports/` est ignoré par git.

In [ ]:
import time

time.sleep(1.1)
avant = set(Path("reports").glob("report-*.json"))
r = run()
apres = set(Path("reports").glob("report-*.json"))
check((len(apres - avant), r.returncode), (1, 0))
check("reports/" in Path(".gitignore").read_text(encoding="utf-8").split(), True)

### 2. `archive_old` — fin 12:40

`archive_old(reports_dir, days)` garde, parmi les entrées de `os.scandir(reports_dir)`, les fichiers dont le nom commence par `report-` et dont `entry.stat().st_mtime` est antérieur à `time.time() - days * 86400`, puis :

```mermaid
flowchart TD
  tri["entrées retenues par os.scandir"] --> aucune(["aucune ?"])
  aucune -- oui --> none["return None"]
  aucune -- non --> dossier["crée le dossier intermédiaire archive-date dans reports_dir"]
  dossier --> move["y déplace chaque rapport retenu : shutil.move"]
  move --> zip["zippe le dossier : shutil.make_archive(str(dossier), 'zip', dossier)"]
  zip --> rm["supprime le dossier intermédiaire : shutil.rmtree"]
  rm --> ret["return chemin du zip"]
```

Le dossier intermédiaire se nomme `archive-<date>`.

`make_archive` ajoute `.zip` au nom de base et renvoie le chemin absolu du zip. `rmtree` ne vise que le dossier que la fonction vient de créer.

**Prédire** — Sans le test `startswith("report-")`, que devient `notes.txt` dans la vérification ci-dessous ?

In [ ]:
# Ma réponse :

**Justifier** — Pourquoi déplacer les rapports dans un dossier intermédiaire avant `make_archive`, au lieu de zipper `reports_dir` directement ?

In [ ]:
# Ma réponse :

Vérification, dans un dossier temporaire : `report-a.json`, `report-b.json` et `notes.txt` sont reculés de 10 jours par `os.utime`, `report-c.json` reste du jour.

In [ ]:
import os
import zipfile

%run -n fleetcheck.py
with tempfile.TemporaryDirectory() as tmp:
    d = Path(tmp)
    for nom in ("report-a.json", "report-b.json", "report-c.json", "notes.txt"):
        (d / nom).write_text("{}", encoding="utf-8")
    vieux = time.time() - 10 * 86400
    for nom in ("report-a.json", "report-b.json", "notes.txt"):
        os.utime(d / nom, (vieux, vieux))

    archive = archive_old(d, 7)
    check(str(archive).endswith(".zip"), True)
    with zipfile.ZipFile(archive) as z:
        check(sorted(z.namelist()), ["report-a.json", "report-b.json"])
    restants = sorted(p.name for p in d.iterdir())
    check([n for n in restants if not n.endswith(".zip")], ["notes.txt", "report-c.json"])
    check(sum(n.endswith(".zip") for n in restants), 1)
    check(any(p.is_dir() for p in d.iterdir()), False)
    check(archive_old(d, 7), None)

### 3. Archivage à chaque run — fin 12:50

À la fin de `main()`, après l'écriture du rapport : `archive_old(config["reports_dir"], config["archive_after_days"])`. Quand des rapports sont archivés, un log `INFO` donne leur nombre et le chemin du zip (texte libre).

Log attendu avec `-v` quand un rapport est archivé (référence v0.7) :

```text
INFO 9 serveur(s) valide(s), 0 ligne(s) invalide(s) dans servers.csv
INFO FLEETCHECK_WEBHOOK_URL absent : pas d'alerte
INFO rapport écrit : reports/report-2026-11-10T124512.json
INFO 1 rapport(s) archivé(s) dans /…/reports/archive-2026-11-10T124512.zip
```

Vérification, sur votre `reports/` : deux runs, le rapport du premier reculé de 10 jours, puis un troisième run : le premier rapport passe dans une archive et quitte `reports/`.

In [ ]:
time.sleep(1.1)
avant = set(Path("reports").glob("report-*.json"))
run()
premier = (set(Path("reports").glob("report-*.json")) - avant).pop()
time.sleep(1.1)
run()
vieux = time.time() - 10 * 86400
os.utime(premier, (vieux, vieux))
time.sleep(1.1)
zips_avant = set(Path("reports").glob("archive-*.zip"))
r = run("-v")
nouveaux = set(Path("reports").glob("archive-*.zip")) - zips_avant
check(len(nouveaux), 1)
with zipfile.ZipFile(nouveaux.pop()) as z:
    check(premier.name in z.namelist(), True)
check((premier.exists(), r.stdout, r.returncode), (False, RESUME, 0))

## Critère final

Trois runs, dont le premier rapport reculé de 10 jours avant le troisième, laissent dans `reports/` deux rapports de plus et une archive qui contient le premier : c'est la vérification de l'étape 3. stdout et code de sortie ne changent pas : le résumé de 4 lignes, code `0`.

## Pour aller plus loin

- `fleetcheck --format json` affiche toujours le rapport sur stdout : comparez-le au dernier fichier de `reports/` (`max(Path("reports").glob("report-*.json"))`, le plus récent grâce au format de date).
- Une archive par run qui archive : au bout d'un an, des dizaines de zips. Proposez une règle de purge des archives et dites quel outil la ferait en entreprise plutôt que `fleetcheck`.